In [4]:
# CELL 1 - Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# CELL 2 - Load data
df = pd.read_csv('Heart_data.csv')

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nClass distribution:")
print(df['HeartDisease'].value_counts())

# Features and target
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Since we only have 500 rows → use ALL of them (no sampling needed)
print(f"\nUsing all {len(df)} samples for training and testing")

# CELL 3 - Train / Test split (80/20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test class balance:  {np.bincount(y_test)}")

# Scale features (needed for Logistic Regression & SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# CELL 4 - 1. Logistic Regression
print("\n" + "="*60)
print("1. LOGISTIC REGRESSION")
print("="*60)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test_scaled))
print(f"Logistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.1f}%)")

# CELL 5 - 2. Random Forest
print("\n" + "="*60)
print("2. RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(n_estimators=300, random_state=42, max_depth=None)
rf.fit(X_train, y_train)  # Trees don't need scaling
acc_rf = accuracy_score(y_test, rf.predict(X_test))
print(f"Random Forest Accuracy:       {acc_rf:.4f} ({acc_rf*100:.1f}%)")

# Show feature importance
print("\nFeature Importance (Random Forest):")
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.round(4))

# CELL 6 - 3. Support Vector Machine
print("\n" + "="*60)
print("3. SUPPORT VECTOR MACHINE (RBF kernel)")
print("="*60)

svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train_scaled, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test_scaled))
print(f"SVM Accuracy:                 {acc_svm:.4f} ({acc_svm*100:.1f}%)")

# CELL 7 - Final Results
print("\n" + "="*60)
print("FINAL ACCURACY COMPARISON (100 test samples)")
print("="*60)
print(f"Logistic Regression   : {acc_lr:.4f}")
print(f"Random Forest         : {acc_rf:.4f}")
print(f"SVM (RBF)             : {acc_svm:.4f}")
print("="*60)

best = max(acc_lr, acc_rf, acc_svm)
if best == acc_rf:
    print("RANDOM FOREST WINS! (as expected with that QuantumPatternFeature)")
elif best == acc_svm:
    print("SVM wins!")
else:
    print("Logistic Regression wins!")

# Optional: Predict on first patient in test set
sample = X_test.iloc[0:1]
pred_rf = rf.predict(sample)[0]
real    = y_test.iloc[0]
print(f"\nExample prediction → True label: {real}, RF predicts: {pred_rf} → {'Correct' if pred_rf == real else 'Wrong'}")

Dataset shape: (500, 7)

First 5 rows:
   Age  Gender  BloodPressure  Cholesterol  HeartRate  QuantumPatternFeature  \
0   68       1            105          191        107               8.362241   
1   58       0             97          249         89               9.249002   
2   44       0             93          190         82               7.942542   
3   72       1             93          183        101               6.495155   
4   37       0            145          166        103               7.653900   

   HeartDisease  
0             1  
1             0  
2             1  
3             1  
4             1  

Class distribution:
HeartDisease
1    300
0    200
Name: count, dtype: int64

Using all 500 samples for training and testing
Training samples: 400
Test samples:     100
Train class balance: [160 240]
Test class balance:  [40 60]

1. LOGISTIC REGRESSION
Logistic Regression Accuracy: 0.9200 (92.0%)

2. RANDOM FOREST
Random Forest Accuracy:       0.9200 (92.0%)

Feature I